# Liniowa Analiza Dyskryminacyjna z Optymalizacją Gradientową (LDA-GO)
![algorithm](pics/algorithm.png)

In [14]:
import numpy as np
from scipy.special import softmax
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder

# Implementacja optymalizatora Adam wykorzystywana w ścieżce NLL 
class AdamOptimizer:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr        # tempo uczenia
        self.beta1 = beta1  # współczynnik zaniku dla momentu 1. rzędu
        self.beta2 = beta2  # współczynnik zaniku dla momentu 2. rzędu
        self.eps = eps      # stała stabilizująca przed dzieleniem przez zero
        self.m = None       # średnia gradientów
        self.v = None       # wariancja gradientów
        self.t = 0          # licznik kroków do korekcji obciążenia

    def step(self, params, grad):
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * grad**2
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


### img_1: pi_hat, mu_hat
![1](pics/1.png) 
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
### img_2: wzory i informacje o standaryzacji wewnątrzklasowej
![2](pics/2.png)
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
### img_3: parametryzacja macierzy precyzji - na tym etapie dokonujemy tylko ustawienia zmiennej d, nie znamy jeszcze L oraz $\sigma^2$
![3](pics/3.png)
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
### img_4: Diagnostyka strukturalna oraz wybór funkcji straty
![4](pics/4.png)

## Objaśnienie obliczania kurtozy
$e_j$ - jest to czwarty moment centralny podzielony przez kwadrat wariancji
$$e_j = \frac{\frac{1}{n} \sum_{i=1}^{n} (R_{ij} - \bar{R}_j)^4}{\left( \frac{1}{n} \sum_{i=1}^{n} (R_{ij} - \bar{R}_j)^2 \right)^2}$$
gdzie:
* $R_{ij}$ — standaryzowana reszta *within-class* dla obserwacji $i$ i cechy $j$,
* $\bar{R}_j$ — średnia reszt dla cechy $j$.

ponieważ w kodzie:
$$R_{ij} = \tilde{X}_{ij} - \tilde{\mu}_{y_i j}$$
to reszty są już wycentrowane względem klas, więc praktycznie:
$$\bar{R}_j \approx 0$$
i wzór upraszcza się do postaci:
$$e_j = \frac{\frac{1}{n}\sum_{i=1}^{n}R_{ij}^4}{\left(\frac{1}{n}\sum_{i=1}^{n}R_{ij}^2\right)^2}$$

# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
### img_5: Ścieżka NLL
![7](pics/7.png)

![5](pics/5.png)

# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
### img_6: Ścieżka CE
![6_1](pics/6_1.png)

![6_2](pics/6_2.png)

$$\delta_k(x_i) = z_i^\top w_k - \frac{1}{2} \|w_k\|^2 + \sigma^2 \left( \tilde{x}_i^\top \tilde{\mu}_k - \frac{1}{2} \|\tilde{\mu}_k\|^2 \right) + \log \hat{\pi}_k$$
Ale wiemy że $\sigma^2 = 0$ dlatego:
$$\delta_k(x_i) = z_i^\top w_k - \frac{1}{2} \|w_k\|^2 + \log \hat{\pi}_k$$
Oraz wiemy że:
* $z_i = L^\top \tilde{x}_i \in \mathbb{R}^d$
* $w_k = L^\top \tilde{\mu}_k \in \mathbb{R}^d$

In [15]:
class LDAGO(BaseEstimator, ClassifierMixin):
    def __init__(self, d=None, max_iter=500, lr=0.01, tol=1e-4, r_thresh=0.1, kappa_thresh=10.0):
        self.d = d                          # d - rząd faktoryzacji macierzy
        self.max_iter = max_iter            # T = 500 - maksymala liczba iteracji
        self.lr = lr                        # lr = 0.01 tempo uczenia dla obu ścieżek
        self.tol = tol                      # próg zbieżności - ‖∇L‖ < 10⁻⁴
        self.r_thresh = r_thresh            # próg rzadkości sygnału, gdy r < 0.1 -> wybierz CE
        self.kappa_thresh = kappa_thresh    # próg kurtozy, gdy k > 10 -> Wybierz CE

    def fit(self, X, y):
        n, p = X.shape                      # n - liczba próbek, p - liczba cech
        le = LabelEncoder()                 
        y_enc = le.fit_transform(y)         # kodowanie etykiet
        self.classes_ = le.classes_         # zapisanie oryginalnych etykiet
        K = len(self.classes_)              # liczba unikalnych klas

        # Estymacja parametrów klas         
        nk = np.bincount(y_enc)             # Liczba próbek w klasie
        pi_hat = nk / n                     # wzór 1 
        mu_hat = np.array([X[y_enc == k].mean(axis=0) for k in range(K)]) # wzór 1

        # ETAP 1 - Standaryzacja wewnątrzklasowa - zgodnie ze wzorami z img_2
        x_bar = X.mean(axis=0)              # średnia globalna - po wszystkich próbkach
        R = X - mu_hat[y_enc]               # reszty wewnątrzklasowe
        s = R.std(axis=0)                   # marignalne wewnątrzklasowe odchylenie standardowe
        s[s < 1e-12] = 1.0                  # zabezpieczenie przed dzieleniem przez 0 w kolejnym kroku
        X_tilde = (X - x_bar) / s           # dane przekształcone
        mu_tilde = (mu_hat - x_bar) / s     # średnie przekształcone

        # ETAP 2 - Parametryzacja macierzy precyzji - ustawienie rządu faktoryzacji d
        d = self.d if self.d is not None else min(20, p)

        # ETAP 3 - Diagnostyka strukturalna oraz wybór funkcji straty
        x_bar_tilde = X_tilde.mean(axis=0)  # średnia z danych przekształconych
        b = np.array([sum(pi_hat[k] * (mu_tilde[k, j] - x_bar_tilde[j])**2 for k in range(K)) for j in range(p)]) # wariancja międzyklasowa
        sum_b = b.sum()                     # suma wariancji międzyklasowej do oblczenia Deff
        sum_b2 = (b**2).sum()               # suma kwadratów wariancji międzyklasowej do oblczenia Deff
        D_eff = (sum_b**2 / sum_b2) if sum_b2 > 1e-12 else 1.0 # obliczenie Deff, zabezpieczenie przed dzieleniem przez 0
        r = D_eff / p                       # obliczenie rzadkości

        # Wytłumaczenie powyżej w sekcji "Objaśnienie obliczania kurtozy"
        R_tilde = X_tilde - mu_tilde[y_enc] 
        e = np.array([np.mean(R_tilde[:, j]**4) / (np.mean(R_tilde[:, j]**2)**2 + 1e-12) for j in range(p)]) #dodano do mianownika małą wartość by nie dzielić prezz 0
        kappa = np.mean(np.abs(e - 3))

        # Wybór CE gdy r<0.1 || k>10, w przeciwnym wypadku wybór NLL
        use_ce = (r < self.r_thresh) or (kappa > self.kappa_thresh)
        
        # ETAP 4 - Optymalizacja gradientowa
        # Przygotowanie CE (Cross Entropy)
        sigma2 = 0.0                        # parametr σ2 = 0 zgodnie z artykułem - zerowanie wariancji izotropowej
        rng = np.random.default_rng(42)     # utworzenie generatora liczb losowych z seedem 42
        L = np.eye(p, d) + rng.normal(0, 0.01, (p, d)) # Inicjalizacja L za pomocą stworzenia macierzy jednostkowej z dodaniem losowego szumu z rozkładu normalnego
        Y_oh = np.eye(K)[y_enc]             # kodowaneie one-hot etykiet klas

        # Ścieżka CE
        if use_ce:
            for _ in range(self.max_iter):
                Z = X_tilde @ L             # Zmienna pomocnicza Z do wzoru
                W = mu_tilde @ L            # Zmienna pomocnicza W do wzoru
                logits = Z @ W.T - 0.5 * np.sum(W**2, axis=1) + np.log(pi_hat + 1e-12) # obliczanie funkcji dyskryminacyjnej
                P = softmax(logits, axis=1) # Zmienna P do wzoru CE
                Res = Y_oh - P              # macierz rzeszt Rik = Yik - Pik
                # Wzór 3.5 CE Path (4)
                term1 = X_tilde.T @ Res @ W 
                term2 = mu_tilde.T @ (Res.T @ Z)
                col_sum = Res.sum(axis=0)   # 1TR
                term3 = mu_tilde.T @ (np.diag(col_sum) @ W)
                grad = -(1/n) * (term1 + term2 - term3) # Ostateczny wzór
                # Metoda spadku gradientowego w celu optymalizacji
                L = L - self.lr * grad
                if np.linalg.norm(grad) < self.tol:     # sprawdza czy próg zbieżności ‖∇L‖ < 10⁻⁴ został osiągnięty przed liczbą iteracji T=500
                    break
                
        # Ścieżka NLL z kurczeniem OAS
        else:
            # Kurczenie kowariancji metodą OAS
            Sw = (R_tilde.T @ R_tilde) / (n - K)        # empiryczna macierz kowariancji wewnątrzklasowej
            
            # Poniższe linie wymagają zaimplementowania algorytmu OAS (OAS, Chen et al., 2010)
            mu_oas = np.trace(Sw) / p
            rho = np.trace(Sw @ Sw)
            alpha_oas = ((1 - 2/p) * rho + np.trace(Sw)**2) / ((n + 1 - 2/p) * (rho - np.trace(Sw)**2 / p))
            alpha_oas = float(np.clip(alpha_oas, 0, 1))
            
            sigma2 = max(1.0 / (alpha_oas * mu_oas + 1e-12), 1e-8)              # wzór z 3.4 sigma2 = 1/(alfa*mu)
            Sw_alpha = (1 - alpha_oas) * Sw + alpha_oas * mu_oas * np.eye(p)    # Σ_tilde w,α = (1 − α)Σ_tilde w + αµI
            adam = AdamOptimizer(lr=self.lr)                                    # Do optymalizacji praca wymienia algorytm Adam z parametrem η = 0.01
            # Algorytm NLL
            for _ in range(self.max_iter):
                # Elementy składowe wzoru
                LtL = L.T @ L
                inner = sigma2 * np.eye(d) + LtL
                inner_inv = np.linalg.inv(inner)
                grad = Sw_alpha @ L - L @ inner_inv # Ostateczny wzór
                L = adam.step(L, grad)              # optymalizacja algorytmem adam
                if np.linalg.norm(grad) < self.tol: # sprawdza czy próg zbieżności ‖∇L‖ < 10⁻⁴ został osiągnięty przed liczbą iteracji T=500
                    break

        # Zachowanie wyuczonych parametrów
        self.L_ = L
        self.sigma2_ = sigma2
        self.mu_tilde_ = mu_tilde
        self.pi_hat_ = pi_hat
        self.x_bar_ = x_bar
        self.s_ = s
        self.K_ = K
        self.use_ce_ = use_ce
        self.r_ = r
        self.kappa_ = kappa
        return self

    # Funckja używana do przewidywania wartości na nauczonym modelu
    def _discriminant(self, X):
        X_tilde = (X - self.x_bar_) / self.s_           # standaryzacja nowych danych
        Z = X_tilde @ self.L_                           # Oblicza paramter Z na podstawie macierzy zoptymalizowanej L
        W = self.mu_tilde_ @ self.L_                    # Oblicza paramter W na podstawie macierzy zoptymalizowanej L
        scores = Z @ W.T - 0.5 * np.sum(W**2, axis=1) + np.log(self.pi_hat_ + 1e-12)  # Oblicza wartości z funkcji dyskryminacyjnej
        # Jeżeli sigma2 > 0 tzn. uczenie ścieżką NLL, należy doliczyć człon izotropowy który w NLL pominięty był bo sigma2=0
        if self.sigma2_ > 0:
            iso = self.sigma2_ * (X_tilde @ self.mu_tilde_.T - 0.5 * np.sum(self.mu_tilde_**2, axis=1))
            scores += iso
        return scores

    def predict(self, X):
        scores = self._discriminant(X)
        return self.classes_[np.argmax(scores, axis=1)]


## Test działania (Opcjonalny)
Możesz wykorzystać poniższą komórkę, aby przetestować działanie modelu na losowo wygenerowanych danych syntetycznych.

In [19]:
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


def run_comparison():
    faces = fetch_olivetti_faces(shuffle=True, random_state=42)
    X_full, y_full = faces.data, faces.target
    X, y = X_full, y_full
    pca = PCA(n_components=0.99, svd_solver='full', random_state=42)
    X_pca = pca.fit_transform(X)
    classifiers = {
        "LDA-GO": LDAGO(),
        "LDA": LinearDiscriminantAnalysis(),
        "LW-LDA": LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
        "LogReg": LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000),
        "NB": GaussianNB(),
        "RF": RandomForestClassifier(n_estimators=100, random_state=42),
        "NN": MLPClassifier(hidden_layer_sizes=(64,), early_stopping=True, max_iter=1000, random_state=42)
    }
    
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    results = {} 
    for name, clf in classifiers.items():
        fold_accuracies = []        
        for train_idx, test_idx in cv.split(X_pca, y):
            X_train, X_test = X_pca[train_idx], X_pca[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            clf.fit(X_train_scaled, y_train)
            preds = clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, preds)
            fold_accuracies.append(acc)
            
        mean_acc = np.mean(fold_accuracies)
        error_rate = (1 - mean_acc) * 100
        results[name] = error_rate
        print(f"{name:<8}: Błąd klasyfikacji = {error_rate:.1f}%")

    print("\n--- PODSUMOWANIE WYNIKÓW ---")
    for name, error in sorted(results.items(), key=lambda item: item[1]):
        print(f"{name:<10}: {error:.1f}%")

if __name__ == "__main__":
    run_comparison()

LDA-GO  : Błąd klasyfikacji = 4.4%
LDA     : Błąd klasyfikacji = 19.1%
LW-LDA  : Błąd klasyfikacji = 4.5%
LogReg  : Błąd klasyfikacji = 21.1%
NB      : Błąd klasyfikacji = 28.5%
RF      : Błąd klasyfikacji = 11.7%
NN      : Błąd klasyfikacji = 93.8%

--- PODSUMOWANIE WYNIKÓW ---
LDA-GO    : 4.4%
LW-LDA    : 4.5%
RF        : 11.7%
LDA       : 19.1%
LogReg    : 21.1%
NB        : 28.5%
NN        : 93.8%
